# Musicm8 — Complete Song + Neural Vocals

Main complete-song workflow. The vocal backend now uses T4-safe short section generation instead of asking ACE-Step to sing the whole song in one pass.

If the instrumental and lyrics are already good, **do not rerun the whole song**: use the `VOCAL-ONLY RETRY` cell below.

In [ ]:
# ============================================================
# MUSICM8 — ONE CLICK COMPLETE SONG
# ============================================================

import os, sys, json, shutil, subprocess
from pathlib import Path

IDEA = "dark UK garage song about knowing a relationship is over but not being able to leave, emotional chords, deep moving bass"
BARS = 32
SEED = 42
VOCALS = True
VOCAL_STYLE = "expressive contemporary lead vocal, intimate verses, emotional hook, clear lyrics, modern UK electronic production"
VOCAL_LANGUAGE = "en"
VOCAL_STEPS = 32
MATCH_ITERS = 48
MATCH_SECONDS = 3.0
FORCE_SOUND_MATCH = False
AI_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

from google.colab import drive
drive.mount("/content/drive", force_remount=False)
ROOT = Path("/content/drive/MyDrive/Musicm8")
AUDIO = ROOT / "audio"
WORK = ROOT / "work"
REPO = Path("/content/Musicm8")
REPO_URL = "https://github.com/Elephant-logic/Musicm8.git"
AUDIO.mkdir(parents=True, exist_ok=True)
WORK.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(WORK / "hf_cache")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "--hard", "origin/main"], check=True)
else:
    shutil.rmtree(REPO, ignore_errors=True)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO)], check=True)

os.chdir(REPO)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-ai.txt"], check=True)
subprocess.run(["apt-get", "update", "-qq"], check=True)
subprocess.run(["apt-get", "install", "-y", "-qq", "espeak-ng"], check=False)

import torch
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No GPU connected. Runtime → Change runtime type → GPU.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU VRAM: %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1024**3))

cmd = [sys.executable, "-u", "ai_producer_workflow.py", "--root", str(ROOT), "--repo", str(REPO), "--idea", IDEA, "--bars", str(BARS), "--seed", str(SEED), "--ai-model", AI_MODEL, "--match-iters", str(MATCH_ITERS), "--match-seconds", str(MATCH_SECONDS), "--vocal-style", VOCAL_STYLE, "--vocal-language", VOCAL_LANGUAGE, "--vocal-steps", str(VOCAL_STEPS)]
if not VOCALS: cmd.append("--no-vocals")
if FORCE_SOUND_MATCH: cmd.append("--force-sound-match")
subprocess.run(cmd, check=True)

PROJECT = WORK / "ai_projects/latest"
from IPython.display import Audio, display
for label, p in [("RAW VOCAL", PROJECT/"vocals/neural_lead_raw.wav"), ("VOCAL MIX", PROJECT/"vocals/vocal_mix.wav"), ("FINAL SONG", PROJECT/"master.wav")]:
    if p.exists():
        print("\n🎵", label)
        display(Audio(str(p)))


## 🎤 VOCAL-ONLY RETRY

Use this after a vocal failure. It keeps the existing instrumental, MIDI, synth matches, song plan and lyrics, pulls the newest vocal code, and retries **only the singer + vocal mix**.

In [ ]:
# ============================================================
# MUSICM8 — VOCAL-ONLY RETRY (NO MUSIC REGENERATION)
# ============================================================
import os, sys, subprocess
from pathlib import Path
from IPython.display import Audio, display

ROOT = Path('/content/drive/MyDrive/Musicm8')
REPO = Path('/content/Musicm8')
PROJECT = ROOT / 'work/ai_projects/latest'
VOCAL_STYLE = "expressive contemporary lead vocal, intimate verses, emotional hook, clear lyrics, modern UK electronic production"
VOCAL_LANGUAGE = 'en'
VOCAL_STEPS = 32
SEED = 42

# Critical: uv temp/cache data must be local Colab storage, not the Drive FUSE mount.
os.environ['UV_CACHE_DIR'] = '/content/musicm8_uv_cache'
os.environ['UV_PYTHON_INSTALL_DIR'] = '/content/musicm8_uv_python'
os.environ['HF_HOME'] = str(ROOT / 'work/hf_cache')
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

subprocess.run(['git','-C',str(REPO),'fetch','--depth','1','origin','main'], check=True)
subprocess.run(['git','-C',str(REPO),'reset','--hard','origin/main'], check=True)
os.chdir(REPO)

print('🎤 Retrying vocals only — existing music will NOT be regenerated')
subprocess.run([sys.executable, '-u', 'retry_vocals.py', '--root', str(ROOT), '--repo', str(REPO), '--style', VOCAL_STYLE, '--language', VOCAL_LANGUAGE, '--steps', str(VOCAL_STEPS), '--seed', str(SEED)], check=True)

RAW = PROJECT/'vocals/neural_lead_raw.wav'
VMIX = PROJECT/'vocals/vocal_mix.wav'
MASTER = PROJECT/'master.wav'
for label, path in [('RAW NEURAL VOCAL',RAW),('PROCESSED VOCAL',VMIX),('FINAL SONG',MASTER)]:
    if path.exists():
        print('\n✅', label)
        display(Audio(str(path)))


## Vocal diagnostics

Only use this if the vocal-only retry still fails. It prints the exact ACE-Step traceback rather than just the wrapper exit code.

In [ ]:
from pathlib import Path
project = Path('/content/drive/MyDrive/Musicm8/work/ai_projects/latest')
status = project/'vocals/vocal_status.json'
log = project/'vocals/vocal_backend.log'
print(status.read_text() if status.exists() else 'No vocal_status.json')
if log.exists():
    print('\n--- EXACT VOCAL BACKEND LOG TAIL ---')
    print('\n'.join(log.read_text(errors='ignore').splitlines()[-120:]))
